# Parameterizing Joint Gaussians

Multivariate normal (Gaussian) distributions are a common default choice for specifying the joint probability distribution over two random vectors. There are a couple of common ways that the joint distribution can be parameterized, and we'll review them in this post.

<!-- TEASER_END -->

## Marginal Parameterization

The most straightforward approach to parameterizing a joint Gaussian is to directly specify the mean vector and covariance matrix.

$$
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix}
\sim
\mathcal{N}
\left(
\begin{bmatrix} m_1 \\ m_2 \end{bmatrix}
,
\begin{bmatrix}
C_1 & C_{12} \\
C_{12}^\top & C_2
\end{bmatrix}
\right)
$$

This is the approach often used, for example, when we use a Gaussian process. The mean function tells us how to compute the mean vector for any collection of values, and the kernel tells us how to compute the covariance matrices.

## Conditional Parameterization

The second way to parameterize a joint normal is to define a marginal distribution for the first component and the conditional distribution of the second component. The marginal distribution is simply a Gaussian with arbitrary mean and covariance:

$$
x_1 \sim \mathcal{N}(m_1, C_1).
$$

The conditional distribution must have a particular structure to keep the joint distribution within the Gaussian family. It needs to be a normal distribution, and the mean of the distribution needs to be a linear function of the first component. We can write this as

$$
x_2 \mid x_1 \sim \mathcal{N} \big( A x_1, C_{1 \mid 2} \big).
$$

Using well-known properties of the multivariate normal, we can work out the marginal distribution over the second component:

$$
x_2 \sim \mathcal{N} \big( A m_1, A C_1 A^\top + C_{1 \mid 2} \big).
$$

The joint distribution is therefore

$$
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix}
\sim
\mathcal{N}
\left(
\begin{bmatrix} m_1 \\ A m_1 \end{bmatrix}
,
\begin{bmatrix}
C_1 & C_{12} \\
C_{12}^\top & A C_1 A^\top + C_{1 \mid 2}
\end{bmatrix}
\right).
$$

We can workout the covariance term using the remaining parameters. First, the expected outer product conditioned on the first component is

$$
\mathbb{E} \big[ x_1 x_2^\top \mid x_1 \big] =
x_1 \mathbb{E} \big[ x_2^\top \mid x_2 \big] =
x_1 x_1^\top A^\top.
$$

Taking the expectation with respect to the first component then gives

$$
\mathbb{E} \big[ x_1 x_2^\top \big] = \big( C_1 + m_1 m_1^\top \big) A^\top.
$$

We then have

$$
C_{12} =
\big( C_1 + m_1 m_1^\top \big) A^\top - m_1 m_1^\top A^\top =
C_1 A^\top.
$$

## What is the Difference?

Why should we use one parameterization over the other? The most obvious answer is that you should use the parameterization that you think you can reasonably describe given domain knowledge. To use Gaussian process regression as an example again, the kernel function allows us to capture inductive biases about how correlated we think two observations should be given no other observations. This very naturally lends itself to the marginal parameterization, since we can directly compute the covariance matrices using the kernel. On the other hand, the conditional distribution allows us to encode a different kind of inductive bias: what is the expected value of the output given inputs?

There are also some computational differences depending on the way you'd like to manipulate the joint distribution. In the marginal parameterization, integrating over one component to get the marginal distribution over the other is trivial. The marginal over the first component is

$$
x_1 \sim \mathcal{N} (m_1, C_1),
$$

and the marginal over the second is

$$
x_2 \sim \mathcal{N} (m_2, C_2).
$$

On the other hand, to get the marginal over $x_2$ using the conditional parameterization requires $O(n_1 n_2)$ flops to compute the mean and $O(n_1 n_2^2)$ operations to compute the covariance.